In [1]:
# Cell 1 - Configuration: choose 'default' (az login), 'device-code', 'aml' (managed identity), or 'keys' for API keys
AUTH_METHOD = 'default'  # change to 'keys' to exercise key-path

# Keep this cell minimal for learners
from dotenv import load_dotenv
import os
load_dotenv("./.env")

print('AUTH_METHOD =', AUTH_METHOD)

AUTH_METHOD = default


In [4]:
# Azure Authentication using Helper Module
import os
from dotenv import load_dotenv
from azure_auth_helper import authenticate_azure

# Load environment variables
load_dotenv("./.env")

# Get required IDs from environment variables
TENANT_ID = os.getenv('AZURE_TENANT_ID')
SUBSCRIPTION_ID = os.getenv('AZURE_SUBSCRIPTION_ID')

if not TENANT_ID:
    raise ValueError("AZURE_TENANT_ID not found in .env file. Please add it to your .env file.")
if not SUBSCRIPTION_ID:
    raise ValueError("AZURE_SUBSCRIPTION_ID not found in .env file. Please add it to your .env file.")

print(f"🏢 Using tenant ID: {TENANT_ID}")
print(f"📋 Using subscription ID: {SUBSCRIPTION_ID}")

# Authenticate with Azure using the configured method
credential = authenticate_azure(
    auth_method='default', 
    tenant_id=TENANT_ID,
    subscription_id=SUBSCRIPTION_ID
)

# Test that the credential actually works
print(f"🔍 Testing credential type: {type(credential).__name__}")
try:
    # Try to get a token to validate the credential
    token = credential.get_token("https://management.azure.com/.default")
    print("✅ Credential test successful!")
    print(f"Token expires: {token.expires_on}")
except Exception as e:
    print(f"❌ Credential test failed: {e}")
    print(f"   Error type: {type(e).__name__}")
    raise

print("🎉 Ready to use Azure AI Foundry!")

🏢 Using tenant ID: 7ec824c6-48d9-4e32-b7a1-9a3df8bdcf66
📋 Using subscription ID: 6e7322e5-6a37-496c-b25c-f647c15bebd8
🔑 Authenticating with Azure CLI using method: default
🏢 Tenant: 7ec824c6-48d9-4e32-b7a1-9a3df8bdcf66
📋 Subscription: 6e7322e5-6a37-496c-b25c-f647c15bebd8
❌ Azure CLI authentication failed: [WinError 2] The system cannot find the file specified


FileNotFoundError: [WinError 2] The system cannot find the file specified

In [ ]:
from azure.identity import DefaultAzureCredential
!az login
credential = DefaultAzureCredential()

c:\Users\hannahhowell\AppData\Local\miniconda3\envs\azure-ai-foundry\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [ ]:
# Cell 2 - Load non-key env vars (always required)
AZURE_TENANT_ID = os.getenv('AZURE_TENANT_ID')
AZURE_SUBSCRIPTION_ID = os.getenv('AZURE_SUBSCRIPTION_ID')
FOUNDRY_API_ENDPOINT = os.getenv('FOUNDRY_API_ENDPOINT') or os.getenv('OPENAI_API_ENDPOINT')
GPT_DEPLOYMENT = os.getenv('GPT4O_DEPLOYMENT_NAME') or os.getenv('GPT_DEPLOYMENT_NAME')
OPENAI_ENDPOINT = os.environ['OPENAI_API_ENDPOINT']


print('Foundry/endpoint:', FOUNDRY_API_ENDPOINT)
print('Tenant:', AZURE_TENANT_ID)
print('Subscription:', AZURE_SUBSCRIPTION_ID)
print('Model/deployment:', GPT_DEPLOYMENT)

In [ ]:
# Cell 3 - Load keys (only when AUTH_METHOD == 'keys')
if AUTH_METHOD == 'keys':
    OPENAI_KEY_= os.environ['OPENAI_API_KEY']

print('Keys loaded:', AUTH_METHOD == 'keys')

In [ ]:
# Cell 4 - Create client and make one simple chat/completion call
from openai import AzureOpenAI
from azure.ai.projects import AIProjectClient

if AUTH_METHOD == 'keys':
    client = AzureOpenAI(
    azure_endpoint=os.environ['OPENAI_API_ENDPOINT'],
    api_key=os.environ['OPENAI_API_KEY'],
    api_version='2023-05-15')
else:
    project = AIProjectClient(
    endpoint=os.getenv("FOUNDRY_API_ENDPOINT"),
    credential=credential)
    
    client = project.get_openai_client(api_version="2024-10-21")
    





In [ ]:
text_prompt = "Should oxford commas always be used?"

response = client.chat.completions.create(
  model=GPT_DEPLOYMENT,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content